# Fiche 2 — Garder un secret : le token d'API

| | |
|---|---|
| 🎯 **Objectif** | Stocker un token dans un fichier `.env` ignoré par git, le lire depuis Python sans jamais l'afficher, et le tester |
| ⏱️ **Durée** | 30 min |
| 🧰 **Prérequis** | Fiche 1 |
| 🔗 **Dans le projet** | Étape 2 (obtenir un token Hugging Face) |

> **L'idée clé.** Un token, c'est **un mot de passe qui coûte de l'argent** : quiconque le lit peut utiliser ton compte et tes crédits. Il vit dans un fichier à part, que git ignore, et ne s'affiche jamais.

## 1. Le principe en trois fichiers

| Fichier | Contenu | Versionné ? |
|---|---|---|
| `.env` | `HF_TOKEN=hf_abc123…` (le vrai token) | ❌ jamais |
| `.gitignore` | une ligne `.env` | ✅ |
| `.env.example` | `HF_TOKEN=hf_xxxxxxxx` (un faux, pour montrer le format) | ✅ |

Pour la démonstration, on travaille dans un **dossier temporaire** avec un **faux** token : rien ne touche ton
vrai `.env`.

In [1]:
import os
import subprocess
import tempfile
from pathlib import Path

from dotenv import load_dotenv

demo = Path(tempfile.mkdtemp())
(demo / ".env").write_text("HF_TOKEN=hf_FAUX_TOKEN_DE_DEMONSTRATION\n")
(demo / ".gitignore").write_text(".env\n")
subprocess.run(["git", "init", "-q", str(demo)])
print(sorted(p.name for p in demo.iterdir()))

['.env', '.git', '.gitignore']


## 2. Lire le token sans l'afficher

`load_dotenv` copie les lignes du fichier dans les variables d'environnement ; `os.environ` les lit.

In [2]:
os.environ.pop("HF_TOKEN", None)  # on repart de zéro pour la démonstration
load_dotenv(demo / ".env")
token = os.environ.get("HF_TOKEN")

# Ce qu'on affiche : la PRÉSENCE du token et sa forme, jamais sa valeur
print("Token trouvé :", token is not None)
print("Commence par hf_ :", token.startswith("hf_"), "| longueur :", len(token))

Token trouvé : True
Commence par hf_ : True | longueur : 30


## 3. Vérifier que git l'ignore vraiment

`git check-ignore` répond par le nom du fichier s'il est ignoré, et ne répond rien sinon.

In [3]:
for fichier in [".env", ".env.example"]:
    (demo / fichier).touch()
    ignore = subprocess.run(["git", "-C", str(demo), "check-ignore", "-q", fichier]).returncode == 0
    print(f"{fichier:<13} ignoré par git : {ignore}")

.env          ignoré par git : True
.env.example  ignoré par git : False


## 4. Tester le token sans rien dépenser

Hugging Face propose une adresse qui répond simplement « à qui appartient ce token ? »
(`https://huggingface.co/api/whoami-v2`). Le package du projet l'emballe dans `api.check_token(token)`, qui renvoie
le **nom du compte** et lève une erreur claire si le token est refusé (code 401). Avec ton vrai `.env` :

```python
from segvet import api
print(api.check_token(api.load_token()))   # affiche le nom du compte, jamais le token
```

Au moment de créer le token sur Hugging Face, donne-lui **le minimum de droits** : un token *fine-grained* avec la
seule permission « Make calls to Inference Providers » suffit. À défaut, un token de type *Read* (lecture seule)
marche aussi. Dans tous les cas : un token qui ne peut pas écrire fait moins de dégâts s'il fuit.

## ⚠️ Piège : le token qui fuit par une sortie de notebook

Le code ne contient pas le token… mais une cellule l'a **affiché**. Les sorties sont enregistrées dans le fichier
`.ipynb` : committer le notebook publie le token. Démonstration avec un notebook miniature :

In [4]:
import json

notebook_fautif = {
    "cells": [{"cell_type": "code", "source": "print(token)", "outputs": [{"output_type": "stream", "text": token}]}]
}
fichier = demo / "analyse.ipynb"
fichier.write_text(json.dumps(notebook_fautif))

# Ce que verrait n'importe qui en lisant le fichier sur GitHub :
contenu = fichier.read_text()
print("Le token apparaît dans le fichier .ipynb :", token in contenu)

Le token apparaît dans le fichier .ipynb : True


**Lecture.** `.gitignore` protège le fichier `.env`, pas les **copies** du token ailleurs. Avant chaque commit, une
recherche suffit : `git grep -n "hf_"` ne doit rien trouver d'autre que `.env.example`. Si un token a fui : on le
**révoque** tout de suite sur Hugging Face (Settings → Access Tokens → Invalidate), puis on en crée un autre ;
effacer le commit ne suffit pas, le dépôt a pu être copié entre-temps.

### ✍️ À toi de jouer

Écris une fonction `cherche_secrets(dossier)` qui parcourt les fichiers `.py` et `.ipynb` d'un dossier et renvoie la liste de ceux qui contiennent une chaîne commençant par `hf_` suivie d'au moins 10 lettres ou chiffres. Teste-la sur `demo`. Indices : `re.search(r"hf_\w{10,}", texte)` cherche un motif dans un texte ; `Path(dossier).rglob("*")` parcourt tous les fichiers d'un dossier.

In [5]:
# Ta réponse ici

<details>
<summary><b>💡 Voir une solution</b> (après avoir cherché !)</summary>

```python
import re

def cherche_secrets(dossier):
    motif = re.compile(r"hf_[A-Za-z0-9_]{10,}")
    return [p.name for p in Path(dossier).rglob("*")
            if p.suffix in {".py", ".ipynb"} and motif.search(p.read_text(errors="ignore"))]

cherche_secrets(demo)   # ['analyse.ipynb']
```

**Pourquoi ?** on cherche la **forme** d'un token, pas sa valeur exacte : la fonction trouve aussi un ancien token oublié. Des outils font cela automatiquement avant chaque commit (gitleaks, detect-secrets).

</details>

## ✅ À retenir

1. Le token vit dans `.env`, listé dans `.gitignore` ; on versionne seulement `.env.example` avec une fausse valeur.
2. On lit le token avec `python-dotenv` et `os.environ` ; on affiche sa présence, **jamais sa valeur**.
3. Droits minimaux à la création ; on le teste avec une requête qui ne coûte rien (`whoami`).
4. Une sortie de notebook peut faire fuir le token : `git grep "hf_"` avant de committer.
5. Token exposé = token **révoqué** immédiatement, pas seulement effacé du dépôt.